# SHM Vision — Kaggle GPU Training (SDNET2018)

Trains the YOLOv8-cls 6-class concrete damage classifier on the **SDNET2018** Kaggle dataset, evaluates it, and produces downloadable artifacts. No code upload needed — project sources are embedded in this notebook.

**Setup (one-time):**
1. New Notebook → **Settings → Accelerator → GPU** (T4 x2 or P100).
2. **Add Input** → attach the SDNET2018 dataset (`structural-defects-network-concrete-crack-images`).
3. Run cells top-to-bottom.

Flow: locate SDNET → normalize folder names → optional out-of-scope class → balanced 80/10/10 split (3200/class, oversampled) → train (yolov8s-cls @ 256px) → evaluate on test → package `best.pt` for download.

**Optional — out-of-scope abstain class:** attach ANY second dataset with diverse
everyday photos (people, sky, grass, cars, wood...) and cell 3b automatically
samples up to 3200 of them into the `z_other` class (oversampled to match the
structural classes) so the model can say "not a structural surface" instead of
forcing a concrete verdict. Skip cell 3b to train the plain 6-class model.

## Step-by-step

1. **Settings** (right panel): Accelerator → **GPU T4 x2** (or P100), Internet → **ON**.
2. **Add Input** → attach SDNET2018 (`structural-defects-network-concrete-crack-images`).
3. *(Optional, for `z_other`)* Add Input → attach any everyday-photos dataset (e.g. Intel Image Classification).
4. **Run all** (or run cells top-to-bottom). Cell 6 (training) takes ~2–4 h on a T4.
5. Watch `training_console.log` / Output tab. After cell 9, download `shm_vision_artifacts/`.
6. Drop `best.pt` into `runs/classify/shm_classification/weights/` at home and `streamlit run app.py`.

See `kaggle/KAGGLE_SETUP.md` for the full guide + resume/troubleshooting.

In [ ]:
# 1. Locate the SDNET2018 dataset under /kaggle/input (auto-discover, slug-agnostic)
from pathlib import Path

CANDIDATES = [
    d for d in Path("/kaggle/input").rglob("*")
    if d.is_dir() and {c.name.lower() for c in d.iterdir() if c.is_dir()} >= {"decks", "pavements", "walls"}
]
assert CANDIDATES, "SDNET2018 not found - add the dataset via Add Input, then re-run."
RAW = CANDIDATES[0]
print("SDNET root:", RAW)
for s in sorted(RAW.iterdir()):
    if s.is_dir():
        print(" ", s.name, "->", [c.name for c in sorted(s.iterdir())])

In [ ]:
# 2. Materialize project code (embedded in this notebook - no upload needed)
import base64
from pathlib import Path

PROJECT = Path("/kaggle/working/shm-vision")
PROJECT.mkdir(parents=True, exist_ok=True)

FILES = {"src/train.py": "...", "scripts/prepare_data.py": "...", "scripts/evaluate.py": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiIKTW9kZWwgRXZhbHVhdGlvbiBTY3JpcHQgZm9yIFNITSBDbGFzc2lmaWNhdGlvbgo9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpDb21wcmVoZW5zaXZlIGV2YWx1YXRpb246IHRvcC0xL3RvcC01IGFjY3VyYWN5LCBmdWxsIGNvbmZ1c2lvbiBtYXRyaXgsCnBlci1jbGFzcyBwcmVjaXNpb24vcmVjYWxsL0YxLCBjYWxpYnJhdGlvbiBkaWFnbm9zdGljcyAoRUNFKSwgYW5kIHNwZWVkCmJlbmNobWFyay4gQWxsIHByZWRpY3Rpb25zIGFyZSBjb2xsZWN0ZWQgZXhwbGljaXRseSAobm8gc3R1YnMpLCBzbyBldmVyeQpyZXBvcnRlZCBtZXRyaWMgaXMgY29tcHV0ZWQgZnJvbSB0aGUgYWN0dWFsIHBlci1pbWFnZSBwcmVkaWN0aW9ucy4KClVzYWdlOgogICAgcHl0aG9uIHNjcmlwdHMvZXZhbHVhdGUucHkgLS13ZWlnaHRzIGJlc3QucHQgLS1kYXRhIGRhdGEvcHJvY2Vzc2VkL3Rlc3QKICAgIHB5dGhvbiBzY3JpcHRzL2V2YWx1YXRlLnB5IC0td2VpZ2h0cyBiZXN0LnB0IC0tZGF0YSBkYXRhL3Byb2Nlc3NlZC90ZXN0IC0tYmVuY2htYXJrCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmltcG9ydCBsb2dnaW5nCmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgRGljdCwgTGlzdAoKaW1wb3J0IG51bXB5IGFzIG5wCmZyb20gdWx0cmFseXRpY3MgaW1wb3J0IFlPTE8KCmxvZ2dpbmcuYmFzaWNDb25maWcoCiAgICBsZXZlbD1sb2dnaW5nLklORk8sIGZvcm1hdD0iJShhc2N0aW1lKXMgLSAlKGxldmVsbmFtZSlzIC0gJShtZXNzYWdlKXMiCikKbG9nZ2VyID0gbG9nZ2luZy5nZXRMb2dnZXIoX19uYW1lX18pCgpJTUFHRV9FWFRFTlNJT05TID0gKCIqLmpwZyIsICIqLmpwZWciLCAiKi5wbmciLCAiKi5ibXAiLCAiKi50aWZmIiwgIioud2VicCIpCgoKZGVmIGNvbGxlY3RfcHJlZGljdGlvbnMoCiAgICBtb2RlbDogWU9MTywKICAgIGRhdGFfcGF0aDogc3RyLAogICAgZGV2aWNlOiBzdHIgPSAiY3B1IiwKKSAtPiB0dXBsZToKICAgICIiIgogICAgUnVuIGluZmVyZW5jZSBvdmVyIGEgZm9sZGVyLXN0cnVjdHVyZWQgKGNsYXNzLXN1YmZvbGRlcikgZGF0YXNldCBhbmQKICAgIGNvbGxlY3QgZ3JvdW5kLXRydXRoIHZzIHByZWRpY3RlZCBsYWJlbHMuCgogICAgQXJnczoKICAgICAgICBtb2RlbDogTG9hZGVkIFlPTE8gY2xhc3NpZmljYXRpb24gbW9kZWwuCiAgICAgICAgZGF0YV9wYXRoOiBSb290IGRpcmVjdG9yeSBjb250YWluaW5nIG9uZSBzdWJmb2xkZXIgcGVyIGNsYXNzLgogICAgICAgIGRldmljZTogVG9yY2ggZGV2aWNlIHNwZWNpZmllci4KCiAgICBSZXR1cm5zOgogICAgICAgICh5X3RydWUsIHlfcHJlZCwgeV9wcm9iLCBjbGFzc19uYW1lcyk6CiAgICAgICAgICAgIHlfdHJ1ZS95X3ByZWQ6IGludGVnZXIgbGFiZWwgYXJyYXlzIGFsaWduZWQgYnkgaW5kZXguCiAgICAgICAgICAgIHlfcHJvYjogKE4sIEMpIHNvZnRtYXggcHJvYmFiaWxpdHkgbWF0cml4LgogICAgICAgICAgICBjbGFzc19uYW1lczogY2xhc3MtaWQgLT4gbmFtZSBtYXBwaW5nIGZyb20gdGhlIG1vZGVsLgoKICAgIFJhaXNlczoKICAgICAgICBGaWxlTm90Rm91bmRFcnJvcjogSWYgbm8gaW1hZ2VzIGFyZSBmb3VuZCB1bmRlciBgYGRhdGFfcGF0aGBgLgogICAgIiIiCiAgICByb290ID0gUGF0aChkYXRhX3BhdGgpCiAgICBpbWFnZV9wYXRoczogTGlzdFtQYXRoXSA9IFtdCiAgICBmb3IgZXh0IGluIElNQUdFX0VYVEVOU0lPTlM6CiAgICAgICAgaW1hZ2VfcGF0aHMuZXh0ZW5kKHJvb3Qucmdsb2IoZXh0KSkKICAgIGltYWdlX3BhdGhzID0gc29ydGVkKGltYWdlX3BhdGhzKQogICAgaWYgbm90IGltYWdlX3BhdGhzOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKAogICAgICAgICAgICBmIk5vIGltYWdlcyBmb3VuZCB1bmRlciB7ZGF0YV9wYXRofSAiCiAgICAgICAgICAgIGYiKGV4cGVjdGVkIDxkYXRhX3BhdGg+LzxjbGFzc19uYW1lPi88aW1hZ2U+KSIKICAgICAgICApCgogICAgIyBHcm91bmQgdHJ1dGggZnJvbSBmb2xkZXIgbmFtZTsgZmFsbCBiYWNrIHRvIGlkIGlmIHVubWFwcGVkLgogICAgbmFtZV90b19pZCA9IHtuYW1lOiBpbnQoaSkgZm9yIGksIG5hbWUgaW4gbW9kZWwubmFtZXMuaXRlbXMoKX0KICAgIHlfdHJ1ZTogTGlzdFtpbnRdID0gW10KICAgIHZhbGlkX3BhdGhzOiBMaXN0W1BhdGhdID0gW10KICAgIGZvciBwIGluIGltYWdlX3BhdGhzOgogICAgICAgIGNsc19uYW1lID0gcC5wYXJlbnQubmFtZS5sb3dlcigpCiAgICAgICAgaWYgY2xzX25hbWUgaW4gbmFtZV90b19pZDoKICAgICAgICAgICAgeV90cnVlLmFwcGVuZChuYW1lX3RvX2lkW2Nsc19uYW1lXSkKICAgICAgICAgICAgdmFsaWRfcGF0aHMuYXBwZW5kKHApCiAgICAgICAgZWxzZToKICAgICAgICAgICAgbG9nZ2VyLndhcm5pbmcoIlNraXBwaW5nICVzOiBmb2xkZXIgJyVzJyBpcyBub3QgYSBrbm93biBjbGFzcyIsIHAsIGNsc19uYW1lKQoKICAgIGxvZ2dlci5pbmZvKCJFdmFsdWF0aW5nICVkIGltYWdlcyBhY3Jvc3MgJWQgY2xhc3Nlcy4uLiIsCiAgICAgICAgICAgICAgICBsZW4odmFsaWRfcGF0aHMpLCBsZW4obW9kZWwubmFtZXMpKQoKICAgIHlfcHJlZDogTGlzdFtpbnRdID0gW10KICAgIHlfcHJvYjogTGlzdFtucC5uZGFycmF5XSA9IFtdCiAgICBmb3IgaSwgcCBpbiBlbnVtZXJhdGUodmFsaWRfcGF0aHMsIDEpOgogICAgICAgIHJlcyA9IG1vZGVsLnByZWRpY3Qoc3RyKHApLCBkZXZpY2U9ZGV2aWNlLCB2ZXJib3NlPUZhbHNlKVswXQogICAgICAgIHlfcHJvYi5hcHBlbmQobnAuYXNhcnJheShyZXMucHJvYnMuZGF0YS5jcHUoKSwgZHR5cGU9bnAuZmxvYXQ2NCkpCiAgICAgICAgeV9wcmVkLmFwcGVuZChpbnQocmVzLnByb2JzLnRvcDEpKQogICAgICAgIGlmIGkgJSAxMDAgPT0gMDoKICAgICAgICAgICAgbG9nZ2VyLmluZm8oIiAgJWQvJWQgaW5mZXJyZWQiLCBpLCBsZW4odmFsaWRfcGF0aHMpKQoKICAgIHJldHVybiAoCiAgICAgICAgbnAuYXJyYXkoeV90cnVlKSwKICAgICAgICBucC5hcnJheSh5X3ByZWQpLAogICAgICAgIG5wLnN0YWNrKHlfcHJvYiksCiAgICAgICAgZGljdChtb2RlbC5uYW1lcyksCiAgICApCgoKZGVmIGV4cGVjdGVkX2NhbGlicmF0aW9uX2Vycm9yKAogICAgeV9wcm9iOiBucC5uZGFycmF5LCB5X3RydWU6IG5wLm5kYXJyYXksIG5fYmluczogaW50ID0gMTUKKSAtPiBmbG9hdDoKICAgICIiIgogICAgRXhwZWN0ZWQgQ2FsaWJyYXRpb24gRXJyb3I6IHxjb25maWRlbmNlIC0gYWNjdXJhY3l8IHdlaWdodGVkIGJ5IGJpbiBtYXNzLgoKICAgIEEgd2VsbC1jYWxpYnJhdGVkIG1vZGVsJ3MgcHJlZGljdGVkIGNvbmZpZGVuY2Ugc2hvdWxkIG1hdGNoIGl0cyBlbXBpcmljYWwKICAgIGFjY3VyYWN5IC0tIGVzc2VudGlhbCB3aGVuIGNvbmZpZGVuY2UgZHJpdmVzIFVOQ0VSVEFJTiBmbGFnZ2luZyBkb3duc3RyZWFtLgoKICAgIEFyZ3M6CiAgICAgICAgeV9wcm9iOiAoTiwgQykgcHJvYmFiaWxpdHkgbWF0cml4LgogICAgICAgIHlfdHJ1ZTogKE4sKSBpbnRlZ2VyIGxhYmVscy4KICAgICAgICBuX2JpbnM6IE51bWJlciBvZiBlcXVhbC13aWR0aCBjb25maWRlbmNlIGJpbnMuCgogICAgUmV0dXJuczoKICAgICAgICBFQ0UgaW4gWzAsIDFdIChsb3dlciBpcyBiZXR0ZXI7IDAgPSBwZXJmZWN0bHkgY2FsaWJyYXRlZCkuCiAgICAiIiIKICAgIGNvbmZpZGVuY2VzID0geV9wcm9iLm1heChheGlzPTEpCiAgICBwcmVkaWN0aW9ucyA9IHlfcHJvYi5hcmdtYXgoYXhpcz0xKQogICAgYWNjdXJhY2llcyA9IChwcmVkaWN0aW9ucyA9PSB5X3RydWUpLmFzdHlwZShucC5mbG9hdDY0KQoKICAgIGVjZSA9IDAuMAogICAgZm9yIGxvIGluIG5wLmxpbnNwYWNlKDAuMCwgMS4wLCBuX2JpbnMgKyAxKVs6LTFdOgogICAgICAgIGhpID0gbG8gKyAxLjAgLyBuX2JpbnMKICAgICAgICBtYXNrID0gKGNvbmZpZGVuY2VzID4gbG8pICYgKGNvbmZpZGVuY2VzIDw9IGhpKQogICAgICAgIGlmIG1hc2suc3VtKCkgPT0gMDoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBlY2UgKz0gbWFzay5tZWFuKCkgKiBhYnMoYWNjdXJhY2llc1ttYXNrXS5tZWFuKCkgLSBjb25maWRlbmNlc1ttYXNrXS5tZWFuKCkpCiAgICByZXR1cm4gZmxvYXQoZWNlKQoKCmRlZiBldmFsdWF0ZV9tb2RlbCgKICAgIHdlaWdodHNfcGF0aDogc3RyLAogICAgZGF0YV9wYXRoOiBzdHIsCiAgICBvdXRwdXRfZGlyOiBzdHIgPSAicnVucy9ldmFsdWF0aW9uIiwKICAgIGRldmljZTogc3RyID0gImNwdSIsCiAgICBiZW5jaG1hcmtfcnVuczogaW50ID0gMCwKKSAtPiBEaWN0OgogICAgIiIiCiAgICBGdWxsIGV2YWx1YXRpb24gcGlwZWxpbmU6IG1ldHJpY3MsIGNvbmZ1c2lvbiBtYXRyaXgsIHBlci1jbGFzcyByZXBvcnQsCiAgICBjYWxpYnJhdGlvbiwgYW5kIG9wdGlvbmFsIHNwZWVkIGJlbmNobWFyay4KCiAgICBBcmdzOgogICAgICAgIHdlaWdodHNfcGF0aDogUGF0aCB0byB0cmFpbmVkIHdlaWdodHMuCiAgICAgICAgZGF0YV9wYXRoOiBEYXRhc2V0IHJvb3QgKGNsYXNzLXN1YmZvbGRlciBsYXlvdXQpLgogICAgICAgIG91dHB1dF9kaXI6IFdoZXJlIGFydGlmYWN0cyAoSlNPTi9DU1YvUE5HKSBhcmUgd3JpdHRlbi4KICAgICAgICBkZXZpY2U6ICdjcHUnIG9yIEdQVSBpZC4KICAgICAgICBiZW5jaG1hcmtfcnVuczogSWYgPiAwLCBydW4gYSBzcGVlZCBiZW5jaG1hcmsgd2l0aCB0aGlzIG1hbnkgcnVucy4KCiAgICBSZXR1cm5zOgogICAgICAgIE1ldHJpY3MgZGljdCAoYWxzbyB3cml0dGVuIHRvIGBgZXZhbHVhdGlvbl9yZXN1bHRzLmpzb25gYCkuCiAgICAiIiIKICAgIGZyb20gc2tsZWFybi5tZXRyaWNzIGltcG9ydCAoCiAgICAgICAgYWNjdXJhY3lfc2NvcmUsCiAgICAgICAgY2xhc3NpZmljYXRpb25fcmVwb3J0LAogICAgICAgIGNvbmZ1c2lvbl9tYXRyaXgsCiAgICAgICAgcHJlY2lzaW9uX3JlY2FsbF9mc2NvcmVfc3VwcG9ydCwKICAgICkKCiAgICBvdXRfZGlyID0gUGF0aChvdXRwdXRfZGlyKQogICAgb3V0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgbG9nZ2VyLmluZm8oIkxvYWRpbmcgbW9kZWw6ICVzIiwgd2VpZ2h0c19wYXRoKQogICAgbW9kZWwgPSBZT0xPKHdlaWdodHNfcGF0aCkKCiAgICB5X3RydWUsIHlfcHJlZCwgeV9wcm9iLCBjbGFzc19uYW1lcyA9IGNvbGxlY3RfcHJlZGljdGlvbnMobW9kZWwsIGRhdGFfcGF0aCwgZGV2aWNlKQoKICAgIHRvcDEgPSBmbG9hdChhY2N1cmFjeV9zY29yZSh5X3RydWUsIHlfcHJlZCkpCiAgICB0b3A1ID0gZmxvYXQobnAubWVhbihbCiAgICAgICAgZ3QgaW4gbnAuYXJnc29ydCgtKHJvdykpWzo1XSBmb3IgZ3QsIHJvdyBpbiB6aXAoeV90cnVlLCB5X3Byb2IpCiAgICBdKSkgaWYgeV9wcm9iLnNoYXBlWzFdID49IDUgZWxzZSBOb25lCgogICAgbG9nZ2VyLmluZm8oIj0iICogNzApCiAgICBsb2dnZXIuaW5mbygiRVZBTFVBVElPTiBSRVNVTFRTIikKICAgIGxvZ2dlci5pbmZvKCI9IiAqIDcwKQogICAgbG9nZ2VyLmluZm8oIlRvcC0xIEFjY3VyYWN5OiAlLjRmIiwgdG9wMSkKICAgIGlmIHRvcDUgaXMgbm90IE5vbmU6CiAgICAgICAgbG9nZ2VyLmluZm8oIlRvcC01IEFjY3VyYWN5OiAlLjRmIiwgdG9wNSkKICAgIGxvZ2dlci5pbmZvKCI9IiAqIDcwKQoKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiAgICAjIENvbmZ1c2lvbiBtYXRyaXggKHJhdyBjb3VudHMgKyByb3ctbm9ybWFsaXplZCkKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiAgICBsYWJlbHMgPSBzb3J0ZWQoY2xhc3NfbmFtZXMua2V5cygpKQogICAgY20gPSBjb25mdXNpb25fbWF0cml4KHlfdHJ1ZSwgeV9wcmVkLCBsYWJlbHM9bGFiZWxzKQogICAgY21fbm9ybSA9IGNtLmFzdHlwZShucC5mbG9hdDY0KSAvIG5wLm1heGltdW0oY20uc3VtKGF4aXM9MSwga2VlcGRpbXM9VHJ1ZSksIDEpCgogICAgIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgICMgUGVyLWNsYXNzIG1ldHJpY3MKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiAgICB0YXJnZXRfbmFtZXMgPSBbY2xhc3NfbmFtZXNbaV0gZm9yIGkgaW4gbGFiZWxzXQogICAgcmVwb3J0X2RpY3QgPSBjbGFzc2lmaWNhdGlvbl9yZXBvcnQoCiAgICAgICAgeV90cnVlLCB5X3ByZWQsIGxhYmVscz1sYWJlbHMsIHRhcmdldF9uYW1lcz10YXJnZXRfbmFtZXMsCiAgICAgICAgb3V0cHV0X2RpY3Q9VHJ1ZSwgemVyb19kaXZpc2lvbj0wLAogICAgKQogICAgcHJlY2lzaW9uLCByZWNhbGwsIGYxLCBzdXBwb3J0ID0gcHJlY2lzaW9uX3JlY2FsbF9mc2NvcmVfc3VwcG9ydCgKICAgICAgICB5X3RydWUsIHlfcHJlZCwgbGFiZWxzPWxhYmVscywgemVyb19kaXZpc2lvbj0wCiAgICApCiAgICBsb2dnZXIuaW5mbygiXG4lcyIsIGNsYXNzaWZpY2F0aW9uX3JlcG9ydCgKICAgICAgICB5X3RydWUsIHlfcHJlZCwgbGFiZWxzPWxhYmVscywgdGFyZ2V0X25hbWVzPXRhcmdldF9uYW1lcywgemVyb19kaXZpc2lvbj0wCiAgICApKQoKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiAgICAjIENhbGlicmF0aW9uCiAgICAjIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgZWNlID0gZXhwZWN0ZWRfY2FsaWJyYXRpb25fZXJyb3IoeV9wcm9iLCB5X3RydWUpCiAgICBsb2dnZXIuaW5mbygiRXhwZWN0ZWQgQ2FsaWJyYXRpb24gRXJyb3I6ICUuNGYiLCBlY2UpCgogICAgcmVzdWx0cyA9IHsKICAgICAgICAibW9kZWwiOiB3ZWlnaHRzX3BhdGgsCiAgICAgICAgImRhdGFzZXQiOiBkYXRhX3BhdGgsCiAgICAgICAgIm5faW1hZ2VzIjogaW50KGxlbih5X3RydWUpKSwKICAgICAgICAibl9jbGFzc2VzIjogaW50KGxlbihsYWJlbHMpKSwKICAgICAgICAidG9wMV9hY2N1cmFjeSI6IHJvdW5kKHRvcDEsIDQpLAogICAgICAgICJ0b3A1X2FjY3VyYWN5Ijogcm91bmQodG9wNSwgNCkgaWYgdG9wNSBpcyBub3QgTm9uZSBlbHNlIE5vbmUsCiAgICAgICAgImV4cGVjdGVkX2NhbGlicmF0aW9uX2Vycm9yIjogcm91bmQoZWNlLCA0KSwKICAgICAgICAibWFjcm9fYXZnIjogewogICAgICAgICAgICAicHJlY2lzaW9uIjogcmVwb3J0X2RpY3RbIm1hY3JvIGF2ZyJdWyJwcmVjaXNpb24iXSwKICAgICAgICAgICAgInJlY2FsbCI6IHJlcG9ydF9kaWN0WyJtYWNybyBhdmciXVsicmVjYWxsIl0sCiAgICAgICAgICAgICJmMS1zY29yZSI6IHJlcG9ydF9kaWN0WyJtYWNybyBhdmciXVsiZjEtc2NvcmUiXSwKICAgICAgICB9LAogICAgICAgICJ3ZWlnaHRlZF9hdmciOiB7CiAgICAgICAgICAgICJwcmVjaXNpb24iOiByZXBvcnRfZGljdFsid2VpZ2h0ZWQgYXZnIl1bInByZWNpc2lvbiJdLAogICAgICAgICAgICAicmVjYWxsIjogcmVwb3J0X2RpY3RbIndlaWdodGVkIGF2ZyJdWyJyZWNhbGwiXSwKICAgICAgICAgICAgImYxLXNjb3JlIjogcmVwb3J0X2RpY3RbIndlaWdodGVkIGF2ZyJdWyJmMS1zY29yZSJdLAogICAgICAgIH0sCiAgICAgICAgInBlcl9jbGFzcyI6IHsKICAgICAgICAgICAgbmFtZTogewogICAgICAgICAgICAgICAgInByZWNpc2lvbiI6IHJvdW5kKGZsb2F0KHApLCA0KSwKICAgICAgICAgICAgICAgICJyZWNhbGwiOiByb3VuZChmbG9hdChyKSwgNCksCiAgICAgICAgICAgICAgICAiZjEtc2NvcmUiOiByb3VuZChmbG9hdChmKSwgNCksCiAgICAgICAgICAgICAgICAic3VwcG9ydCI6IGludChzKSwKICAgICAgICAgICAgfQogICAgICAgICAgICBmb3IgbmFtZSwgcCwgciwgZiwgcyBpbiB6aXAodGFyZ2V0X25hbWVzLCBwcmVjaXNpb24sIHJlY2FsbCwgZjEsIHN1cHBvcnQpCiAgICAgICAgfSwKICAgICAgICAiY29uZnVzaW9uX21hdHJpeCI6IHsKICAgICAgICAgICAgImxhYmVscyI6IHRhcmdldF9uYW1lcywKICAgICAgICAgICAgImNvdW50cyI6IGNtLnRvbGlzdCgpLAogICAgICAgICAgICAicm93X25vcm1hbGl6ZWQiOiBucC5yb3VuZChjbV9ub3JtLCA0KS50b2xpc3QoKSwKICAgICAgICB9LAogICAgfQoKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiAgICAjIE9wdGlvbmFsIHNwZWVkIGJlbmNobWFyawogICAgIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgIGlmIGJlbmNobWFya19ydW5zID4gMDoKICAgICAgICBzYW1wbGUgPSBzdHIobmV4dCgKICAgICAgICAgICAgcCBmb3IgZXh0IGluIElNQUdFX0VYVEVOU0lPTlMgZm9yIHAgaW4gUGF0aChkYXRhX3BhdGgpLnJnbG9iKGV4dCkKICAgICAgICApKQogICAgICAgIGZvciBfIGluIHJhbmdlKDEwKTogICMgd2FybXVwCiAgICAgICAgICAgIG1vZGVsLnByZWRpY3Qoc2FtcGxlLCBkZXZpY2U9ZGV2aWNlLCB2ZXJib3NlPUZhbHNlKQogICAgICAgIHRpbWVzID0gW10KICAgICAgICBmb3IgXyBpbiByYW5nZShiZW5jaG1hcmtfcnVucyk6CiAgICAgICAgICAgIHQwID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgICAgICBtb2RlbC5wcmVkaWN0KHNhbXBsZSwgZGV2aWNlPWRldmljZSwgdmVyYm9zZT1GYWxzZSkKICAgICAgICAgICAgdGltZXMuYXBwZW5kKHRpbWUucGVyZl9jb3VudGVyKCkgLSB0MCkKICAgICAgICByZXN1bHRzWyJzcGVlZF9iZW5jaG1hcmsiXSA9IHsKICAgICAgICAgICAgImF2ZXJhZ2VfbXMiOiByb3VuZChmbG9hdChucC5tZWFuKHRpbWVzKSkgKiAxMDAwLCAyKSwKICAgICAgICAgICAgInN0ZF9tcyI6IHJvdW5kKGZsb2F0KG5wLnN0ZCh0aW1lcykpICogMTAwMCwgMiksCiAgICAgICAgICAgICJ0aHJvdWdocHV0X2ZwcyI6IHJvdW5kKDEuMCAvIGZsb2F0KG5wLm1lYW4odGltZXMpKSwgMiksCiAgICAgICAgICAgICJydW5zIjogYmVuY2htYXJrX3J1bnMsCiAgICAgICAgfQogICAgICAgIGxvZ2dlci5pbmZvKAogICAgICAgICAgICAiU3BlZWQ6ICUuMmYgwrEgJS4yZiBtcyAoJS4yZiBpbWcvcykiLAogICAgICAgICAgICByZXN1bHRzWyJzcGVlZF9iZW5jaG1hcmsiXVsiYXZlcmFnZV9tcyJdLAogICAgICAgICAgICByZXN1bHRzWyJzcGVlZF9iZW5jaG1hcmsiXVsic3RkX21zIl0sCiAgICAgICAgICAgIHJlc3VsdHNbInNwZWVkX2JlbmNobWFyayJdWyJ0aHJvdWdocHV0X2ZwcyJdLAogICAgICAgICkKCiAgICAjIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgIyBBcnRpZmFjdHMKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiAgICB3aXRoIG9wZW4ob3V0X2RpciAvICJldmFsdWF0aW9uX3Jlc3VsdHMuanNvbiIsICJ3IiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICBqc29uLmR1bXAocmVzdWx0cywgZiwgaW5kZW50PTIpCgogICAgIyBQZXItY2xhc3MgQ1NWIGZvciBzcHJlYWRzaGVldHMuCiAgICB3aXRoIG9wZW4ob3V0X2RpciAvICJwZXJfY2xhc3NfbWV0cmljcy5jc3YiLCAidyIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAgZi53cml0ZSgiY2xhc3MscHJlY2lzaW9uLHJlY2FsbCxmMSxzdXBwb3J0XG4iKQogICAgICAgIGZvciBuYW1lIGluIHRhcmdldF9uYW1lczoKICAgICAgICAgICAgcGMgPSByZXN1bHRzWyJwZXJfY2xhc3MiXVtuYW1lXQogICAgICAgICAgICBmLndyaXRlKAogICAgICAgICAgICAgICAgZiJ7bmFtZX0se3BjWydwcmVjaXNpb24nXX0se3BjWydyZWNhbGwnXX0se3BjWydmMS1zY29yZSddfSx7cGNbJ3N1cHBvcnQnXX1cbiIKICAgICAgICAgICAgKQoKICAgICMgQ29uZnVzaW9uIG1hdHJpeCBwbG90IChtYXRwbG90bGliIGF2YWlsYWJsZSB3aXRoIHVsdHJhbHl0aWNzKS4KICAgIHRyeToKICAgICAgICBpbXBvcnQgbWF0cGxvdGxpYgogICAgICAgIG1hdHBsb3RsaWIudXNlKCJBZ2ciKQogICAgICAgIGltcG9ydCBtYXRwbG90bGliLnB5cGxvdCBhcyBwbHQKCiAgICAgICAgZmlnLCBheCA9IHBsdC5zdWJwbG90cyhmaWdzaXplPSg4LCA2KSkKICAgICAgICBpbSA9IGF4Lmltc2hvdyhjbV9ub3JtLCBjbWFwPSJCbHVlcyIsIHZtaW49MCwgdm1heD0xKQogICAgICAgIGF4LnNldF94dGlja3MocmFuZ2UobGVuKHRhcmdldF9uYW1lcykpKQogICAgICAgIGF4LnNldF95dGlja3MocmFuZ2UobGVuKHRhcmdldF9uYW1lcykpKQogICAgICAgIGF4LnNldF94dGlja2xhYmVscyh0YXJnZXRfbmFtZXMsIHJvdGF0aW9uPTQ1LCBoYT0icmlnaHQiLCBmb250c2l6ZT04KQogICAgICAgIGF4LnNldF95dGlja2xhYmVscyh0YXJnZXRfbmFtZXMsIGZvbnRzaXplPTgpCiAgICAgICAgZm9yIGkgaW4gcmFuZ2UobGVuKGxhYmVscykpOgogICAgICAgICAgICBmb3IgaiBpbiByYW5nZShsZW4obGFiZWxzKSk6CiAgICAgICAgICAgICAgICBheC50ZXh0KGosIGksIGYie2NtW2ksIGpdfVxue2NtX25vcm1baSwgal06LjAlfSIsCiAgICAgICAgICAgICAgICAgICAgICAgIGhhPSJjZW50ZXIiLCB2YT0iY2VudGVyIiwgZm9udHNpemU9NywKICAgICAgICAgICAgICAgICAgICAgICAgY29sb3I9IndoaXRlIiBpZiBjbV9ub3JtW2ksIGpdID4gMC41IGVsc2UgImJsYWNrIikKICAgICAgICBheC5zZXRfeGxhYmVsKCJQcmVkaWN0ZWQiKQogICAgICAgIGF4LnNldF95bGFiZWwoIlRydWUiKQogICAgICAgIGF4LnNldF90aXRsZShmIkNvbmZ1c2lvbiBNYXRyaXggKHRvcC0xID0ge3RvcDE6LjElfSkiKQogICAgICAgIGZpZy5jb2xvcmJhcihpbSkKICAgICAgICBmaWcudGlnaHRfbGF5b3V0KCkKICAgICAgICBmaWcuc2F2ZWZpZyhvdXRfZGlyIC8gImNvbmZ1c2lvbl9tYXRyaXgucG5nIiwgZHBpPTE1MCkKICAgICAgICBwbHQuY2xvc2UoZmlnKQogICAgICAgIGxvZ2dlci5pbmZvKCJTYXZlZCBjb25mdXNpb25fbWF0cml4LnBuZyIpCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICBsb2dnZXIud2FybmluZygiTWF0cGxvdGxpYiBwbG90IHNraXBwZWQ6ICVzIiwgZXhjKQoKICAgIGxvZ2dlci5pbmZvKCJSZXN1bHRzIHNhdmVkIHRvICVzIiwgb3V0X2RpciAvICJldmFsdWF0aW9uX3Jlc3VsdHMuanNvbiIpCiAgICByZXR1cm4gcmVzdWx0cwoKCmRlZiBtYWluKCkgLT4gTm9uZToKICAgICIiIkNMSSBlbnRyeXBvaW50LiIiIgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249IkV2YWx1YXRlIFNITSBDbGFzc2lmaWNhdGlvbiBNb2RlbCIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXdlaWdodHMiLCB0eXBlPXN0ciwgcmVxdWlyZWQ9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0iUGF0aCB0byBtb2RlbCB3ZWlnaHRzICgucHQpIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZGF0YSIsIHR5cGU9c3RyLCBkZWZhdWx0PSJkYXRhL3Byb2Nlc3NlZC90ZXN0IiwKICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0iRGF0YXNldCByb290IHdpdGggY2xhc3Mgc3ViZm9sZGVycyIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dHB1dCIsIHR5cGU9c3RyLCBkZWZhdWx0PSJydW5zL2V2YWx1YXRpb24iLAogICAgICAgICAgICAgICAgICAgICAgICBoZWxwPSJPdXRwdXQgZGlyZWN0b3J5IGZvciByZXN1bHRzIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZGV2aWNlIiwgdHlwZT1zdHIsIGRlZmF1bHQ9ImNwdSIsCiAgICAgICAgICAgICAgICAgICAgICAgIGhlbHA9IkRldmljZSAoJ2NwdScgb3IgR1BVIGlkKSIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWJlbmNobWFyayIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIsCiAgICAgICAgICAgICAgICAgICAgICAgIGhlbHA9IlJ1biBpbmZlcmVuY2Ugc3BlZWQgYmVuY2htYXJrIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcnVucyIsIHR5cGU9aW50LCBkZWZhdWx0PTEwMCwKICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0iQmVuY2htYXJrIHJ1bnMgKHdpdGggLS1iZW5jaG1hcmspIikKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCgogICAgZXZhbHVhdGVfbW9kZWwoCiAgICAgICAgd2VpZ2h0c19wYXRoPWFyZ3Mud2VpZ2h0cywKICAgICAgICBkYXRhX3BhdGg9YXJncy5kYXRhLAogICAgICAgIG91dHB1dF9kaXI9YXJncy5vdXRwdXQsCiAgICAgICAgZGV2aWNlPWFyZ3MuZGV2aWNlLAogICAgICAgIGJlbmNobWFya19ydW5zPWFyZ3MucnVucyBpZiBhcmdzLmJlbmNobWFyayBlbHNlIDAsCiAgICApCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=", "config/hyperparams.yaml": "IyBIeXBlcnBhcmFtZXRlcnMgZm9yIFlPTE92OCBDbGFzc2lmaWNhdGlvbiAtIFN0cnVjdHVyYWwgSGVhbHRoIE1vbml0b3JpbmcNCiMgT3B0aW1pemVkIGZvciBtdWx0aS1jbGFzcyBjb25jcmV0ZSBkYW1hZ2UgY2xhc3NpZmljYXRpb24gKERlY2svUGF2ZW1lbnQvV2FsbCB4IENyYWNrZWQvVW5jcmFja2VkKQ0KDQojIE1vZGVsIENvbmZpZ3VyYXRpb24NCm1vZGVsOg0KICBiYXNlX3dlaWdodHM6ICd5b2xvdjhzLWNscy5wdCcgICMgU21hbGw6IG1vcmUgY2FwYWNpdHkgdGhhbiBuYW5vIGZvciBoYWlybGluZSBjcmFja3MNCiAgIyBPcHRpb25zOiB5b2xvdjhuLWNscy5wdCAoZmFzdGVzdCksIHlvbG92OHMtY2xzLnB0LCB5b2xvdjhtLWNscy5wdCAobW9zdCBhY2N1cmF0ZSkNCiAgcHJldHJhaW5lZDogdHJ1ZSAgICAgICAgICAgICAgICAjIFVzZSBJbWFnZU5ldCBwcmUtdHJhaW5lZCB3ZWlnaHRzDQogIHRhc2s6ICdjbGFzc2lmaWNhdGlvbicgICAgICAgICAgIyBFeHBsaWNpdCB0YXNrIGRlZmluaXRpb24NCg0KIyBUcmFpbmluZyBQYXJhbWV0ZXJzDQp0cmFpbmluZzoNCiAgZXBvY2hzOiAxNTAgICAgICAgICAgICAgICAgICAgICMgTWF4aW11bSB0cmFpbmluZyBlcG9jaHMgKGNsYXNzaWZpY2F0aW9uIGNvbnZlcmdlcyBmYXN0ZXIpDQogIGltZ3N6OiAyNTYgICAgICAgICAgICAgICAgICAgICAgIyBOYXRpdmUgU0RORVQgdGlsZSBzaXplOyAzMjArIG9ubHkgdXAtc2FtcGxlcyAobm8gbmV3IGRldGFpbCwgc2xvd2VyIGVwb2NocykNCiAgYmF0Y2g6IDY0ICAgICAgICAgICAgICAgICAgICAgICAjIExhcmdlciBiYXRjaCBmb3IgY2xhc3NpZmljYXRpb24gKGZpdHMgbW9yZSBpbiBtZW1vcnkpDQogIHBhdGllbmNlOiAzMCAgICAgICAgICAgICAgICAgICAgICMgRWFybHkgc3RvcHBpbmcgcGF0aWVuY2U6IGdpdmUgbm9pc3kgdmFsLWxvc3Mgc3Bpa2VzIHJvb20gdG8gcmVjb3Zlcg0KICBkZXZpY2U6ICdhdXRvJyAgICAgICAgICAgICAgICAgICMgJ2F1dG8nID0gR1BVIGlmIENVREEgYXZhaWxhYmxlLCBlbHNlIENQVQ0KICB3b3JrZXJzOiAnYXV0bycgICAgICAgICAgICAgICAgICMgJ2F1dG8nID0gMCBvbiBXaW5kb3dzLCA0IG9uIExpbnV4L0thZ2dsZQ0KICBjYWNoZTogdHJ1ZSAgICAgICAgICAgICAgICAgICAgICMgQ2FjaGUgaW1hZ2VzIGluIFJBTSAofjE4ME1CIGZvciAxNmsgaW1hZ2VzKQ0KICBzaW5nbGVfY2xzOiBmYWxzZSAgICAgICAgICAgICAgIyBNdWx0aS1jbGFzcyBjbGFzc2lmaWNhdGlvbg0KDQojIE9wdGltaXplciBTZXR0aW5ncw0Kb3B0aW1pemVyOg0KICBuYW1lOiAnQWRhbVcnICAgICAgICAgICAgICAgICAgICMgT3B0aW1pemVyIHR5cGUNCiAgbHIwOiAwLjAwMSAgICAgICAgICAgICAgICAgICAgICAjIEluaXRpYWwgbGVhcm5pbmcgcmF0ZQ0KICBscmY6IDAuMDEgICAgICAgICAgICAgICAgICAgICAgICMgRmluYWwgTFIgPSBscjAgKiBscmYNCiAgbW9tZW50dW06IDAuOTM3ICAgICAgICAgICAgICAgICAjIFNHRCBtb21lbnR1bSAvIEFkYW0gYmV0YTENCiAgd2VpZ2h0X2RlY2F5OiAwLjAwMDUgICAgICAgICAgICAjIFdlaWdodCBkZWNheQ0KICB3YXJtdXBfZXBvY2hzOiAzLjAgICAgICAgICAgICAgICMgV2FybXVwIGVwb2Nocw0KICB3YXJtdXBfbW9tZW50dW06IDAuOCAgICAgICAgICAgICMgV2FybXVwIGluaXRpYWwgbW9tZW50dW0NCiAgY29zX2xyOiB0cnVlICAgICAgICAgICAgICAgICAgICAjIENvc2luZSBkZWNheTogc21vb3RoZXIgdGFpbCB0aGFuIGxpbmVhciwgZmV3ZXIgbGF0ZSBzcGlrZXMNCg0KIyBBdWdtZW50YXRpb24gZm9yIENvbmNyZXRlIEltYWdlcw0KYXVnbWVudGF0aW9uOg0KICBoc3ZfaDogMC4wMTUgICAgICAgICAgICAgICAgICAgICMgSFNWIEh1ZQ0KICBoc3ZfczogMC43ICAgICAgICAgICAgICAgICAgICAgICMgSFNWIFNhdHVyYXRpb24NCiAgaHN2X3Y6IDAuNCAgICAgICAgICAgICAgICAgICAgICAjIEhTViBWYWx1ZQ0KICBkZWdyZWVzOiAxNS4wICAgICAgICAgICAgICAgICAgICAjIFJvdGF0aW9uICgrLy0gZGVncmVlcykgLSBpbXBvcnRhbnQgZm9yIGFuZ2xlZCBwaG90b3MNCiAgdHJhbnNsYXRlOiAwLjEgICAgICAgICAgICAgICAgICAgIyBUcmFuc2xhdGlvbg0KICBzY2FsZTogMC41ICAgICAgICAgICAgICAgICAgICAgICAjIFNjYWxlDQogIHNoZWFyOiAyLjAgICAgICAgICAgICAgICAgICAgICAgICMgU2hlYXINCiAgZmxpcHVkOiAwLjUgICAgICAgICAgICAgICAgICAgICAgIyBWZXJ0aWNhbCBmbGlwOiBjcmFja3MgaGF2ZSBubyBjYW5vbmljYWwgdXAtZGlyZWN0aW9uDQogIGZsaXBscjogMC41ICAgICAgICAgICAgICAgICAgICAgIyBIb3Jpem9udGFsIGZsaXANCiAgbWl4dXA6IDAuMSAgICAgICAgICAgICAgICAgICAgICAjIE1peFVwIGF1Z21lbnRhdGlvbg0KICAjIEVyYXNpbmcvcmFuZC1hdWdtZW50IGNhbiB3aXBlIG91dCB0aGUgdGhpbiBjcmFjayB0ZXh0dXJlIHRoYXQNCiAgIyBkaXN0aW5ndWlzaGVzIGNyYWNrZWQvdW5jcmFja2VkOyBrZWVwIGJvdGggbG93IGZvciB0aGlzIGRvbWFpbi4NCiAgZXJhc2luZzogMC4yICAgICAgICAgICAgICAgICAgICAgIyBSYW5kb20gZXJhc2luZyAod2FzIDAuNCkNCiAgYXV0b19hdWdtZW50OiBudWxsICAgICAgICAgICAgICAgIyBSYW5kQXVnbWVudCBkaXN0b3J0cyBzdWJ0bGUgY3JhY2sgY3Vlcw0KDQojIE5PVEUgb24gcmV0cmFpbmluZyBhZnRlciB0aGUgMjAyNi0wOSBLYWdnbGUgcnVuICh0b3AxPTAuNzc4LCBiZXN0IGVwb2NoIDcvMjcpOg0KIyAgICogZGVja19jcmFja2VkIHJlY2FsbCB3YXMgMC41NiAoaGFpcmxpbmUgbWlzc2VzIC0+IHByZWRpY3RlZCB1bmNyYWNrZWQpOg0KIyAgICAga2VlcCBpbWdzeiBhdCBTRE5FVC1uYXRpdmUgMjU2IChubyBmYWtlIGRldGFpbCBmcm9tIHVwc2NhbGluZykgYW5kIHJlbHkNCiMgICAgIG9uIGZsaXBzICsgbG9uZ2VyIHBhdGllbmNlIGluc3RlYWQ7IGNhcGFjaXR5IChzLWNscykgaXMgdGhlIGxldmVyLg0KIyAgICogU3BsaXQgZGF0YSBieSBTT1VSQ0UgKHZpZGVvL3Nlc3Npb24pLCBub3QgcGVyLWltYWdlOiBuZWFyLWR1cGxpY2F0ZQ0KIyAgICAgZnJhbWVzIG9mIHRoZSBzYW1lIHN1cmZhY2UgYWNyb3NzIHRyYWluL3ZhbCBpbmZsYXRlIHRlc3QgYWNjdXJhY3kNCiMgICAgIHdoaWxlIHJlYWwtd29ybGQgKG91dC1vZi1kaXN0cmlidXRpb24pIHBob3RvcyBmYWlsLg0KIyAgICogel9vdGhlciBtdXN0IGJlIG92ZXJzYW1wbGVkIHRvIG1hdGNoIHRoZSBzdHJ1Y3R1cmFsIGNsYXNzZXMgb3IgdGhlDQojICAgICBtb2RlbCBhbG1vc3QgbmV2ZXIgcHJlZGljdHMgaXQgKHJlY2FsbCAwLjI3NSB3aGVuIHRyYWluZWQgYXQgfjYyJQ0KIyAgICAgb2YgdGhlIG90aGVyIGNsYXNzZXMnIGJ1ZGdldCkuDQoNCiMgTG9zcyBGdW5jdGlvbiAoaGFuZGxlIGNsYXNzIGltYmFsYW5jZSkNCmxvc3M6DQogIGxhYmVsX3Ntb290aGluZzogMC4wNSAgICAgICAgICAgICMgTG93ZXIgdGhhbiAwLjE6IGtlZXAgbG9naXQgbWFyZ2lucyBmb3Igc3VidGxlIGNyYWNrZWQvdW5jcmFja2VkIGN1ZXMNCiAgIyBOT1RFOiBjdXJyZW50IGRhdGFzZXQgaXMgYWxyZWFkeSBiYWxhbmNlZCAoMTYwMC0zMjAwIGltZ3MvY2xhc3MsIHNlZQ0KICAjIGRhdGEvcHJvY2Vzc2VkL2RhdGFzZXRfc3RhdGlzdGljcy5qc29uKSAtLSBjbGFzcyB3ZWlnaHRzIHVubmVjZXNzYXJ5Lg0KICAjIFVsdHJhbHl0aWNzIGNsYXNzaWZ5IG1vZGUgaGFzIG5vIGBjbGFzc193ZWlnaHRzYCBhcmcgZWl0aGVyOyBiYWxhbmNlDQogICMgdmlhIG92ZXJzYW1wbGluZyBpbiBzY3JpcHRzL3ByZXBhcmVfZGF0YS5weSBpZiBmdXR1cmUgZGF0YSBza2V3cy4NCg0KIyBWYWxpZGF0aW9uICYgTG9nZ2luZw0KdmFsaWRhdGlvbjoNCiAgdmFsX2ludGVydmFsOiAxICAgICAgICAgICAgICAgICAgIyBWYWxpZGF0ZSBldmVyeSBlcG9jaA0KICBzYXZlX3BlcmlvZDogMTAgICAgICAgICAgICAgICAgICMgU2F2ZSBjaGVja3BvaW50IGV2ZXJ5IE4gZXBvY2hzDQogIHByb2plY3Q6ICdydW5zL2NsYXNzaWZ5JyAgICAgICAgICMgT3V0cHV0IGRpcmVjdG9yeQ0KICBuYW1lOiAnc2htX2NsYXNzaWZpY2F0aW9uJyAgICAgICMgRXhwZXJpbWVudCBuYW1lDQogIGV4aXN0X29rOiBmYWxzZSAgICAgICAgICAgICAgICAgIyBPdmVyd3JpdGUgZXhpc3RpbmcNCiAgdmVyYm9zZTogdHJ1ZSAgICAgICAgICAgICAgICAgICAgIyBWZXJib3NlIG91dHB1dA0KICBwbG90czogdHJ1ZSAgICAgICAgICAgICAgICAgICAgICAjIEdlbmVyYXRlIHRyYWluaW5nIHBsb3RzDQogICMgQ29uZnVzaW9uIG1hdHJpeA0KICBjb25mX21hdHJpeDogdHJ1ZSAgICAgICAgICAgICAgICAjIEdlbmVyYXRlIGNvbmZ1c2lvbiBtYXRyaXgNCg0KIyBTdHJ1Y3R1cmFsIEhlYWx0aCBNb25pdG9yaW5nIFNwZWNpZmljDQpzaG06DQogIHN0cnVjdHVyZV90eXBlczogW2RlY2ssIHBhdmVtZW50LCB3YWxsXQ0KICBjb25kaXRpb25zOiBbY3JhY2tlZCwgdW5jcmFja2VkXQ0KICAjIFNhZmV0eSB0aHJlc2hvbGRzIHBlciBzdHJ1Y3R1cmUgdHlwZSAoY3JhY2sgc2V2ZXJpdHkgaW4gbW0pDQogIHNhZmV0eV90aHJlc2hvbGRzOg0KICAgIGRlY2s6DQogICAgICBjcml0aWNhbF9jcmFja193aWR0aDogMC4zDQogICAgICBtYXhfYWxsb3dhYmxlOiAwLjUNCiAgICBwYXZlbWVudDoNCiAgICAgIGNyaXRpY2FsX2NyYWNrX3dpZHRoOiA2LjANCiAgICAgIG1heF9hbGxvd2FibGU6IDEyLjANCiAgICB3YWxsOg0KICAgICAgY3JpdGljYWxfY3JhY2tfd2lkdGg6IDAuMw0KICAgICAgbWF4X2FsbG93YWJsZTogMS4wDQogICMgQ2FtZXJhIGRlZmF1bHRzIGZvciBwaHlzaWNhbCBtZWFzdXJlbWVudA0KICBkZWZhdWx0X2Rpc3RhbmNlX21tOiAyMDAwICAgICAgICAjIDIgbWV0ZXJzDQogIGRlZmF1bHRfZm9jYWxfbGVuZ3RoX3B4OiA4MDANCiAgIyBNaW5pbXVtIGNvbmZpZGVuY2UgZm9yIHN0cnVjdHVyYWwgYXNzZXNzbWVudA0KICBtaW5fY29uZmlkZW5jZTogMC43NQ0K"}

for rel, b64 in FILES.items():  # ponytail: base64 blob for zero-quoting-bug embedding; view sources in the repo
    dest = PROJECT / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(base64.b64decode(b64))
    print("wrote", dest)


In [ ]:
# 2b. Environment check: fail fast if GPU missing
import sys

import torch, torchvision, ultralytics

print(f"python {sys.version.split()[0]} | torch {torch.__version__} | "
      f"torchvision {torchvision.__version__} | ultralytics {ultralytics.__version__}")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise SystemExit("CUDA NOT AVAILABLE - enable GPU accelerator in Notebook Settings, then re-run.")

In [ ]:
# 3. Normalize SDNET naming (Decks/Cracked/Non-cracked) -> prepare_data.py layout (deck/cracked/uncracked)
#    Symlinks: zero-copy, works because prepare_data.py only reads through them.
import os

NORM = Path("/kaggle/working/sdnet_normalized")
MAP = {"decks": "deck", "pavements": "pavements", "walls": "walls"}

if NORM.exists():
    import shutil; shutil.rmtree(NORM)

for src_d in sorted(RAW.iterdir()):
    if not src_d.is_dir():
        continue
    structure = MAP.get(src_d.name.lower())
    if structure is None:
        continue
    for cond_d in sorted(src_d.iterdir()):
        if not cond_d.is_dir():
            continue
        low = cond_d.name.lower()
        if "non" in low:
            cond = "uncracked"
        elif "crack" in low:
            cond = "cracked"
        else:
            continue
        dest = NORM / structure / cond
        dest.parent.mkdir(parents=True, exist_ok=True)
        try:
            os.symlink(cond_d, dest)
        except OSError:
            import shutil
            shutil.copytree(cond_d, dest, dirs_exist_ok=True)
        print(dest, "->", cond_d)
print("Normalized tree ready.")

In [ ]:
# 3b. (OPTIONAL) Build the out-of-scope "z_other" abstain class.
#     Attaches any OTHER image dataset (everyday photos: sky, grass, people,
#     cars, wood...) so the model can abstain on non-structural surfaces.
#     Budget matches the structural classes (~3200): with fewer samples the
#     model almost never predicts z_other (observed recall 0.275 @ 2000).
#     prepare_data --balance --min-per-class fills any residual gap by
#     oversampling. Delete/ignore this cell to train the plain 6-class model.
import random

MAX_OTHER = 3200
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

other_pool = []
for ds in sorted(Path("/kaggle/input").iterdir()):
    if ds.resolve() == RAW.resolve():
        continue  # skip SDNET itself
    imgs = [p for p in ds.rglob("*") if p.suffix.lower() in IMG_EXT and p.is_file()]
    if imgs:
        other_pool.extend(imgs)

if other_pool:
    random.seed(42)
    sample = random.sample(other_pool, min(MAX_OTHER, len(other_pool)))
    dest = NORM / "other" / "mixed"
    dest.mkdir(parents=True, exist_ok=True)
    for i, p in enumerate(sample):
        try:
            os.symlink(p, dest / f"{i:05d}_{p.name}")
        except OSError:
            import shutil
            shutil.copy2(p, dest / f"{i:05d}_{p.name}")
    print(f"z_other class: linked {len(sample)} diverse images from {len({p.parent for p in sample})} folders")
else:
    print("No extra dataset attached - training the plain 6-class model (fine).")

In [ ]:
# 4. Build data/processed: balanced 80/10/10 split, max 3200 images/class (~16k train / 2k / 2k)
import subprocess, sys

cmd = [
    sys.executable, str(PROJECT / "scripts/prepare_data.py"),
    "--raw", str(NORM),
    "--output", str(PROJECT / "data/processed"),
    "--balance", "--max-per-class", "3200", "--min-per-class", "3200",
    "--seed", "42",
]
p = subprocess.run(cmd, capture_output=True, text=True)
print(p.stdout[-3000:])
if p.returncode != 0:
    print(p.stderr[-2000:])
assert p.returncode == 0, "prepare_data failed - see output above."
DATA = PROJECT / "data/processed"

In [ ]:
# 5. Verify splits + class names match config.py CLASS_NAMES (YOLO sorts folders alphabetically;
#    deck_* < pavement_* < wall_* < z_other, so ids 0-5 always match config; id 6 = z_other only
#    when cell 3b produced that class)
import sys
from pathlib import Path

sys.path.insert(0, str(PROJECT / "src"))
sys.path.insert(0, str(PROJECT))  # for config import inside train.py
EXPECTED = {"train", "val", "test"}
BASE_CLASSES = ["deck_cracked", "deck_uncracked", "pavement_cracked",
                "pavement_uncracked", "wall_cracked", "wall_uncracked"]
ALLOWED = set(BASE_CLASSES) | {"z_other"}

for split in sorted(EXPECTED):
    classes = sorted(p.name for p in (DATA / split).iterdir() if p.is_dir())
    n = sum(len(list((DATA / split / c).glob("*"))) for c in classes)
    assert set(classes) <= ALLOWED and classes == sorted(classes), f"{split}: unexpected class folders: {classes}"
    print(f"{split}: {n} images, classes={classes}")
print("Dataset verified. Class id order matches config.py CLASS_NAMES.")

In [ ]:
# 6. TRAINING - 150 epochs, batch 64, patience 20, device auto (GPU), workers auto (4 on Linux)
#    T4: ~2-4 h. Console streamed to /kaggle/working/training_console.log
#    Resume after an interrupted session: keep last.pt in shm_vision_artifacts, set RESUME=True.
import subprocess, sys, time

EPOCHS = 150
BATCH = 64
RESUME = False

if RESUME:
    cmd = [sys.executable, str(PROJECT / "src/train.py"),
           "--resume", str(PROJECT / "runs/classify/shm_classification/weights/last.pt")]
else:
    cmd = [
        sys.executable, str(PROJECT / "src/train.py"),
        "--data", str(DATA),
        "--config", str(PROJECT / "config/hyperparams.yaml"),
        "--epochs", str(EPOCHS),
        "--batch", str(BATCH),
        "--name", "shm_classification",
    ]

print(" ".join(cmd))
t0 = time.time()
with open("/kaggle/working/training_console.log", "a") as log:
    p = subprocess.run(cmd, cwd=PROJECT, stdout=log, stderr=subprocess.STDOUT)
print(f"train exit code: {p.returncode} after {(time.time() - t0) / 60:.1f} min")
if p.returncode != 0:
    print(open("/kaggle/working/training_console.log").read()[-3000:])
assert p.returncode == 0, "Training failed - see training_console.log."

In [ ]:
# 7. Training curves from ultralytics results.csv
import pandas as pd
import plotly.express as px

run_dir = PROJECT / "runs/classify/shm_classification"
df = pd.read_csv(run_dir / "results.csv")
df.columns = [c.strip() for c in df.columns]

metrics = [m for m in ("train/loss", "val/loss", "metrics/accuracy_top1", "metrics/accuracy_top5")
           if m in df.columns]
px.line(df, x="epoch", y=metrics, title="Training curves").show()

In [ ]:
# 8. Evaluate best.pt on the test split (top-1/top-5, per-class P/R/F1, confusion matrix, ECE)
import subprocess, sys

cmd = [
    sys.executable, str(PROJECT / "scripts/evaluate.py"),
    "--weights", str(PROJECT / "runs/classify/shm_classification/weights/best.pt"),
    "--data", str(DATA / "test"),
    "--output", str(PROJECT / "runs/evaluation"),
    "--device", "0",
]
p = subprocess.run(cmd, capture_output=True, text=True)
print(p.stdout[-4000:])
if p.stderr:
    print(p.stderr[-2000:])
assert p.returncode == 0, "Evaluation failed - see output above."

In [ ]:
# 9. Package artifacts for download via the notebook Output tab
import shutil

ART = Path("/kaggle/working/shm_vision_artifacts")
ART.mkdir(exist_ok=True)

run_dir = PROJECT / "runs/classify/shm_classification"
for f in [
    run_dir / "weights/best.pt",
    run_dir / "weights/last.pt",
    run_dir / "results.csv",
    run_dir / "results.png",
    run_dir / "confusion_matrix.png",
    PROJECT / "runs/evaluation/evaluation_results.json",
    PROJECT / "runs/evaluation/per_class_metrics.csv",
    PROJECT / "runs/evaluation/confusion_matrix.png",
    "/kaggle/working/training_console.log",
]:
    if Path(f).exists():
        shutil.copy2(f, ART / Path(f).name)
    else:
        print("missing (skipped):", f)

print("Artifacts:", sorted(p.name for p in ART.iterdir()))

In [ ]:
# 10. Smoke test: reload best.pt and predict on one test image
from ultralytics import YOLO

best = YOLO(PROJECT / "runs/classify/shm_classification/weights/best.pt")
img = next((DATA / "test" / "wall_cracked").glob("*.jpg"))
r = best.predict(str(img), device=0, verbose=False)[0]
name = best.names[int(r.probs.top1)]
print(f"{img.name}: {name} (conf={float(r.probs.top1conf):.3f})")
print("Model classes:", best.names)
print("Model reload + inference OK. Download shm_vision_artifacts from the Output tab.")